# 04b -- LGD validation

**What this notebook does (plain English):** Part 5 of the framework (APS 113
Validation paras 1-6; APG 113 para 140's eight elements; WP14 Section IV) says an
LGD model must be **independently validated**, just like a PD model. The repo
already validates PD (notebook 03c + PSI) but had **no LGD validation** -- the most
visible gap to a credit-risk reviewer. This notebook closes it, mirroring the style
of `03c_PD_OutOfTime_Validation.ipynb`:

1. **Out-of-time / out-of-regime** -- fit the two-stage LGD on the crisis vintages
   and predict the calm one, then the reverse.
2. **Predicted-vs-realised backtest at cohort level** -- by predicted-LGD decile,
   not loan-by-loan.
3. **Discrimination** -- how well predicted severity rank-orders realised severity.
4. **Stability** -- drop each vintage in turn and watch the estimate move.
5. **Benchmarking note** -- because internal data is thin, benchmarking and
   qualitative review carry more weight than backtesting (APG 113 para 140(c); WP14).

**Headline result:** the LGD model **rank-orders** severity but, like PD, its
**level is regime-dependent** -- a model trained only on the calm 2015 book badly
**under-predicts** downturn severity, which is exactly why a downturn LGD is used.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# LGD is validated only on defaulted, DISPOSED loans (the loans with a real
# settled loss). Reuse the same two-stage model the production notebook 04 uses.
import pandas as pd
import numpy as np
from src.models import TwoStageLGD
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
disposed = base[base['disposed'] & base['lgd'].notna()].copy()
print('disposed defaults available for LGD validation:', len(disposed))
print('by vintage:'); print(disposed['vintage_year'].value_counts().sort_index())

disposed defaults available for LGD validation: 13466
by vintage:
vintage_year
2006    4066
2007    4479
2008    2134
2009     543
2010     496
2011     379
2012     379
2013     305
2014     202
2015     136
2016      91
2017      93
2018      62
2019      36
2020      16
2021      12
2022      37
Name: count, dtype: int64


In [3]:
# One out-of-time split: fit the LGD model on the TRAIN vintages only, then
# predict the held-out TEST vintage. No leakage -- the test year never trains.
def oot_lgd(train_years, test_years, label):
    tr = disposed[disposed['vintage_year'].isin(train_years)]
    te = disposed[disposed['vintage_year'].isin(test_years)]
    model = TwoStageLGD().fit(tr)
    pred = model.predict(te)
    return {
        'split': label,
        'n_test': len(te),
        'observed_lgd': round(float(te['lgd'].mean()), 4),
        'predicted_lgd': round(float(np.mean(pred)), 4),
        'pred_minus_obs': round(float(np.mean(pred) - te['lgd'].mean()), 4),
    }

In [4]:
# Out-of-regime tests, the reverse 'what-if', and the genuine FORWARD holdout (R3-V3):
# fit on pre-2020 and score the never-seen 2020-2022 disposed defaults cold. The forward
# LGD test is thin (few recent workouts are fully resolved), so it is read alongside the
# cross-regime splits rather than on its own.
oot_rows = [
    oot_lgd([2007, 2008], [2015], 'A) train crisis 2007+08 -> test calm 2015'),
    oot_lgd([2015], [2007, 2008], 'B) reverse: train calm 2015 -> test crisis (under-predicts)'),
    oot_lgd(list(range(2006, 2020)), [2020, 2021, 2022], 'C) FORWARD holdout: train pre-2020 -> test 2020-22 (thin)'),
]
oot = pd.DataFrame(oot_rows)
oot

,split,n_test,observed_lgd,predicted_lgd,pred_minus_obs
0,A) train crisis 2007+08 -> test calm 2015,136,0.2464,0.4326,0.1862
1,B) reverse: train calm 2015 -> test crisis (un...,6613,0.5673,0.2081,-0.3591
2,C) FORWARD holdout: train pre-2020 -> test 202...,65,0.3212,0.3292,0.0080


In [5]:
# Cohort backtest: fit on everything, bucket disposed defaults by PREDICTED-LGD
# decile, and compare mean predicted vs mean realised in each bucket (cohort-level,
# never loan-by-loan -- WP14 warns point-in-time realised LGD is noisy per loan).
full = TwoStageLGD().fit(disposed)
disposed['lgd_hat'] = full.predict(disposed)
disposed['pred_decile'] = pd.qcut(disposed['lgd_hat'], 10, duplicates='drop', labels=False) + 1
backtest = disposed.groupby('pred_decile').agg(
    n=('lgd', 'size'),
    mean_predicted=('lgd_hat', 'mean'),
    mean_realised=('lgd', 'mean'),
).reset_index().round(4)
backtest['gap'] = (backtest['mean_predicted'] - backtest['mean_realised']).round(4)
backtest

,pred_decile,n,mean_predicted,mean_realised,gap
0,1,1347,0.2975,0.2892,0.0083
1,2,1347,0.3956,0.4236,-0.0280
2,3,1346,0.4554,0.4869,-0.0315
3,4,1347,0.5081,0.5027,0.0054
4,5,1346,0.5427,0.5201,0.0226
5,6,1347,0.5689,0.5437,0.0252
6,7,1346,0.5911,0.5557,0.0354
7,8,1347,0.6116,0.5797,0.0319
8,9,1346,0.6332,0.6351,-0.0019
9,10,1347,0.6633,0.7441,-0.0808


In [6]:
# R3-LGD5: realised-vs-predicted CALIBRATION by an INDEPENDENT risk segment (original-LTV
# band) -- the LGD analogue of the PD calibration test (APS 113 Att D Validation para 3;
# APG 113 para 140 element 3). Calibration is judged WITHIN business-recognised segments,
# not just by the model's own output decile, so a reviewer can see where it over/under-shoots.
disposed['ltv_band'] = pd.cut(
    pd.to_numeric(disposed['original_ltv'], errors='coerce'),
    [0, 60, 70, 80, 90, 200], labels=['<=60', '60-70', '70-80', '80-90', '90+'])
seg_cal = disposed.groupby('ltv_band', observed=True).agg(
    n=('lgd', 'size'),
    realised_lgd=('lgd', 'mean'),
    predicted_lgd=('lgd_hat', 'mean'),
).reset_index().round(4)
seg_cal['gap_pred_minus_real'] = (seg_cal['predicted_lgd'] - seg_cal['realised_lgd']).round(4)
save_csv(seg_cal, 'outputs/tables/04b_lgd_calibration_by_segment.csv')
seg_cal

,ltv_band,n,realised_lgd,predicted_lgd,gap_pred_minus_real
0,<=60,848,0.4848,0.5661,0.0813
1,60-70,1524,0.5770,0.5509,-0.0261
2,70-80,6000,0.5868,0.5404,-0.0464
3,80-90,2106,0.5209,0.5375,0.0166
4,90+,2959,0.4016,0.4696,0.0680


In [7]:
# R3-LGD5: external BENCHMARKING of the modelled LGD against published mortgage
# severities (qualitative anchors -- APG 113 para 140(c) / WP14 Section IV). These are
# documented reference ranges, NOT fitted, used only to sanity-check the level is plausible.
from src import definitions as d_def
down_lgd = float(disposed.loc[d_def.is_downturn_vintage(disposed['vintage_year']), 'lgd'].mean())
calm_lgd = float(disposed.loc[~d_def.is_downturn_vintage(disposed['vintage_year']), 'lgd'].mean())
bench = pd.DataFrame([
    {'source': 'This model -- downturn (GFC) realised LGD', 'lgd_ref': round(down_lgd, 3),
     'note': 'GFC 2006-09 disposed defaults'},
    {'source': 'This model -- calm/other realised LGD', 'lgd_ref': round(calm_lgd, 3),
     'note': 'non-GFC vintages'},
    {'source': 'APRA APS 113 retail-mortgage LGD floor', 'lgd_ref': 0.20,
     'note': 'regulatory minimum where own-LGD not approved (Att B)'},
    {'source': 'Published US GFC residential severities (indicative)', 'lgd_ref': '0.40-0.60',
     'note': 'distressed dispositions 2008-2011 -- reference range, not fitted'},
])
save_csv(bench, 'outputs/tables/04b_lgd_benchmarking.csv')
bench

,source,lgd_ref,note
0,This model -- downturn (GFC) realised LGD,0.565,GFC 2006-09 disposed defaults
1,This model -- calm/other realised LGD,0.342,non-GFC vintages
2,APRA APS 113 retail-mortgage LGD floor,0.2,regulatory minimum where own-LGD not approved ...
3,Published US GFC residential severities (indic...,0.40-0.60,distressed dispositions 2008-2011 -- reference...


In [8]:
# Discrimination on the loss-only loans: does higher predicted severity line up
# with higher realised severity? Spearman rank correlation + R^2.
loss_only = disposed[disposed['lgd'] > 0.05]
spearman = float(loss_only['lgd_hat'].corr(loss_only['lgd'], method='spearman'))
ss_res = float(((loss_only['lgd'] - loss_only['lgd_hat']) ** 2).sum())
ss_tot = float(((loss_only['lgd'] - loss_only['lgd'].mean()) ** 2).sum())
r2 = 1 - ss_res / ss_tot
print(f'Spearman(predicted, realised) on loss-only loans: {spearman:.3f}')
print(f'R^2 of predicted vs realised severity            : {r2:.3f}')

Spearman(predicted, realised) on loss-only loans: 0.327
R^2 of predicted vs realised severity            : 0.086


In [9]:
# Stability: re-fit dropping each vintage in turn and see how the overall mean
# predicted LGD moves -- the 'stability analysis' WP14 asks for.
all_pred = float(TwoStageLGD().fit(disposed).predict(disposed).mean())
stab_rows = [{'configuration': 'all vintages', 'mean_predicted_lgd': round(all_pred, 4),
              'shift_vs_all': 0.0}]
for y in sorted(disposed['vintage_year'].unique()):
    sub = disposed[disposed['vintage_year'] != y]
    mp = float(TwoStageLGD().fit(sub).predict(sub).mean())
    stab_rows.append({'configuration': f'drop {y}', 'mean_predicted_lgd': round(mp, 4),
                      'shift_vs_all': round(mp - all_pred, 4)})
stability = pd.DataFrame(stab_rows)
stability

,configuration,mean_predicted_lgd,shift_vs_all
0,all vintages,0.5267,0.0000
1,drop 2006,0.5032,-0.0236
2,drop 2007,0.5014,-0.0253
3,drop 2008,0.5238,-0.0030
4,drop 2009,0.5314,0.0046
5,drop 2010,0.5321,0.0053
6,drop 2011,0.5312,0.0045
7,drop 2012,0.5326,0.0059
8,drop 2013,0.5304,0.0037
9,drop 2014,0.5294,0.0027


In [10]:
# Combine the headline validation results into one saved table.
val = pd.concat([
    oot.assign(section='out_of_time').rename(columns={'split': 'detail'})[
        ['section', 'detail', 'n_test', 'observed_lgd', 'predicted_lgd', 'pred_minus_obs']],
    stability.assign(section='stability', n_test=np.nan).rename(
        columns={'configuration': 'detail', 'mean_predicted_lgd': 'predicted_lgd'})[
        ['section', 'detail', 'n_test', 'predicted_lgd']],
    pd.DataFrame([{'section': 'discrimination', 'detail': 'spearman / R2 on loss-only',
                   'observed_lgd': round(spearman, 4), 'predicted_lgd': round(r2, 4)}]),
], ignore_index=True)
save_csv(val, 'outputs/tables/04b_lgd_validation.csv')
val

,section,detail,n_test,observed_lgd,predicted_lgd,pred_minus_obs
0,out_of_time,A) train crisis 2007+08 -> test calm 2015,136.0,0.2464,0.4326,0.1862
1,out_of_time,B) reverse: train calm 2015 -> test crisis (un...,6613.0,0.5673,0.2081,-0.3591
2,out_of_time,C) FORWARD holdout: train pre-2020 -> test 202...,65.0,0.3212,0.3292,0.0080
3,stability,all vintages,NaN,NaN,0.5267,NaN
4,stability,drop 2006,NaN,NaN,0.5032,NaN
5,stability,drop 2007,NaN,NaN,0.5014,NaN
6,stability,drop 2008,NaN,NaN,0.5238,NaN
7,stability,drop 2009,NaN,NaN,0.5314,NaN
8,stability,drop 2010,NaN,NaN,0.5321,NaN
9,stability,drop 2011,NaN,NaN,0.5312,NaN


## Interpretation (plain English)

- **Out-of-time / out-of-regime.** A model trained on the **crisis** books
  **over-predicts** the calm 2015 book (predicted ~43% vs realised ~25%) -- i.e. it is
  *conservative* out-of-regime, which is the safe direction. The **reverse** is the
  dangerous one: training only on calm 2015 and predicting the crisis **under-predicts**
  downturn severity badly (predicted ~21% vs realised ~57%). A model built only in good
  times is blind to a downturn; this is the headline out-of-time finding and the reason
  the downturn LGD (notebook 04 / 06) is used for the conservative estimate.
- **Forward holdout (R3-V3).** Fitting on **pre-2020** and scoring the never-seen
  **2020-2022** loans cold is the honest production test. For LGD it is deliberately
  read with caution -- only a handful of recent defaults are fully worked out -- so the
  cross-regime splits and the cohort backtest carry the weight; the PD forward holdout
  (notebook 03c, split D) is the stronger of the two because it has far more test loans.
- **Cohort backtest.** Read by predicted-LGD decile, mean predicted and mean realised
  track in the same direction -- the model is **calibrated in rank**. Per the WP14
  caveat, this is a cohort comparison; a single point-in-time realised LGD must **not**
  be compared directly to a long-run estimate loan-by-loan.
- **Segment calibration (R3-LGD5).** Realised vs predicted LGD is also compared **within
  original-LTV bands** (an independent risk segment, not the model's own output): the
  `gap` column shows the model tracks realised severity across LTV without a systematic
  bias -- the LGD analogue of the PD calibration test (APS 113 Att D Validation para 3).
- **Discrimination.** A positive Spearman correlation between predicted and realised
  severity on the loss-only loans confirms the model **rank-orders** loss size, though
  mortgage LGD is inherently noisy so the R^2 is modest -- normal for severity models.
- **Stability.** Dropping any single vintage moves the overall mean predicted LGD only
  modestly, **except** when the crisis volume is removed, which pulls the estimate down
  -- consistent with severity being driven by the downturn cohorts.

## Benchmarking note (APG 113 para 140(c); WP14)

The internal sample is now far deeper -- **17 vintages and ~13k disposed defaults** across
a full cycle -- so backtesting carries real weight, and the `04b_lgd_benchmarking.csv` table
anchors the level against external references. The modelled **downturn (GFC) severity ~56%**
sits squarely within **published US agency mortgage loss severities** for 2008-2011 (broadly
~40-60% on distressed dispositions), and the **calm/other ~34%** and the **APRA 20% floor**
bracket it sensibly. Where the sample is still thin -- the **2020-2022 workouts not yet fully
resolved** -- benchmarking and the incomplete-workout sensitivity (notebook 04) carry more
weight than the raw recent realised numbers. In production this external comparison would be
refreshed annually alongside an expert-judgement overlay.